#Data Export



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/2019-Oct.csv')
df.head()

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/2019-Oct.csv', nrows=100000)

df.head()

In [ ]:
df.info()

# 📊 Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

In [ ]:
df['event_time'] = pd.to_datetime(df['event_time'])

In [ ]:
df = df.sort_values(by=['user_id', 'event_time'])

In [ ]:
df = df.drop_duplicates()

# 📉 Funnel Analysis

In [ ]:
funnel_map = {
    'view': 1,
    'cart': 2,
    'purchase': 3
}

df['stage'] = df['event_type'].map(funnel_map)

In [ ]:
funnel = df.groupby('event_type')['user_id'].nunique()
print(funnel)

we take out the coversion rate by eg:view-> cart [formula - next/current]
DroffRate : 1- Conversion rate


In [ ]:

view_users = set(df[df['event_type'] == 'view']['user_id'])
cart_users = set(df[df['event_type'] == 'cart']['user_id'])
purchase_users = set(df[df['event_type'] == 'purchase']['user_id'])


cart_users_filtered = view_users.intersection(cart_users)
purchase_users_filtered = cart_users_filtered.intersection(purchase_users)


view_count = len(view_users)
cart_count = len(cart_users_filtered)
purchase_count = len(purchase_users_filtered)

print(view_count, cart_count, purchase_count)

In [ ]:
cart_rate = cart_count / view_count
purchase_rate = purchase_count / cart_count

print("View → Cart:", cart_rate)
print("Cart → Purchase:", purchase_rate)

# 🧪 A/B Testing

Identified a critical drop-off (95%) at the product interaction stage, indicating low engagement during initial user interaction. However, observed strong conversion (60%) from cart to purchase, suggesting high purchase intent once users enter the checkout funnel. Recommended optimizing product pages, pricing strategies, and user experience to improve top-of-funnel conversion.

In [ ]:
import numpy as np

df['group'] = np.where(df['user_id'] % 2 == 0, 'A', 'B')

df[['user_id','group']].head()

In [ ]:
total_users = df.groupby('group')['user_id'].nunique()
print(total_users)

In [ ]:
purchase_users = df[df['event_type'] == 'purchase'].groupby('group')['user_id'].nunique()
print(purchase_users)

In [ ]:
conversion_rate = purchase_users / total_users
print(conversion_rate)


In [ ]:
from statsmodels.stats.proportion import proportions_ztest

success = purchase_users.values
total = total_users.values

z_stat, p_val = proportions_ztest(success, total)

print("Z-stat:", z_stat)
print("p-value:", p_val)


In [ ]:
if p_val < 0.05:
    print("Significant difference → Choose better group")
else:
    print("No significant difference → No strong winner")

In [ ]:
print("Conversion A:", conversion_rate['A'])
print("Conversion B:", conversion_rate['B'])


# 📈 Visualization




In [ ]:
import matplotlib.pyplot as plt

stages = ['View', 'Cart', 'Purchase']
counts = [view_count, cart_count, purchase_count]

plt.figure()
plt.bar(stages, counts)
plt.title("User Funnel Analysis")
plt.xlabel("Stages")
plt.ylabel("Number of Users")
plt.show()

In [ ]:
groups = ['A', 'B']
values = [conversion_rate['A'], conversion_rate['B']]

plt.figure()
plt.bar(groups, values)
plt.title("A/B Testing Conversion Comparison")
plt.ylabel("Conversion Rate")
plt.show()

In [ ]:
top_users = df.groupby('user_id').size().reset_index(name='events')
top_users = top_users.sort_values(by='events', ascending=False).head(10)

print(top_users)

In [ ]:
revenue = df[df['event_type'] == 'purchase']['price'].sum()
print("Total Revenue:", revenue)

In [ ]:
event_counts = df['event_type'].value_counts()
print(event_counts)

# 💼 Business Insights

## 🚨 Major Drop at Top of Funnel
- Around 95% users drop between view and cart stage  
- Indicates low engagement at product level  

## ✅ Strong Purchase Intent
- ~60% users complete purchase after adding to cart  
- Checkout process is efficient  

## 🎯 Key Bottleneck
- Main issue lies in converting views → cart  

## 🧪 A/B Testing Result
- Variant B has slightly higher conversion (7.18% vs 6.69%)  
- Difference is not statistically significant (p = 0.246)  

## ⚠️ Data Quality Insight
- Observed inconsistencies in funnel tracking  
- Indicates potential issues in event logging  

# 🚀 Business Recommendations

- Improve product page UX and design  
- Add trust signals (reviews, ratings)  
- Optimize pricing and offers  
- Run longer and more impactful A/B tests  